In [17]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import json
import time

In [18]:
def load_api_key():
    """
    Load Steam API keys from the .env file located in the .venv folder.
    
    Returns:
        tuple: A tuple containing (api_key).
    """
    # Notebook is in the 'src/' folder, so go up one level to reach '.venv/.env'
    env_path = r'..\src\config.env'
    
    load_dotenv(dotenv_path=env_path)
    api_key = os.getenv('itd_api_key')
    return api_key

api_key = load_api_key()


In [19]:
games = pd.read_json(r'..\data\games_id_all.json')
games = games.loc['apps', 'response']

def name_formatting(id):
    game_data = games[id]
    #print(games.loc['apps', 'response'][id])
    game_name = game_data['name']
    game_id = game_data['appid']
    formatted_name = game_name.lower().replace(' ', '-').replace(':', '')
    return formatted_name, game_id

In [22]:
for index, game in enumerate(games):

    if(index == 1000):          #temporary stop
        break
    if(index % 80):
        time.sleep(60)

    URL = 'https://api.isthereanydeal.com/games/search/v1'

    game_name, app_id = name_formatting(index)
    new_entry = {"appid": app_id, "name": game_name}

    params = {
        'key': api_key,
        'title': game_name         #fetching game name to get uuid
    }

    response = requests.get(URL, params=params)

    if response.status_code == 200:
        data = response.json()

        game_id = data[0]["id"]

        URL_price = 'https://api.isthereanydeal.com/games/history/v2'       #fetching  price data

        params = {
            'key': api_key,
            'id': game_id,
            'shops': 61,
            'country': 'PL'
            }

        response = requests.get(URL_price, params=params)

        if response.status_code == 200:
            data = response.json()
            new_entry.update({'currency': data[0]['deal']['price']['currency']})
            for i in data:
                del i['shop']
                del i['deal']['regular'] 
                del i['deal']['price']['currency']               #necessary data
            
            new_entry.update({'price_changes': data})
            with open(r"..\data\games_prices\price_data.jsonl", "a", encoding="utf-8") as jsonl_file:
                jsonl_file.write(json.dumps(new_entry) + "\n")
                

            with open(r"..\data\games_prices\used_ids.txt", "a", encoding="utf-8") as id_file:
                id_file.write(str(app_id) + "\n")
            
        else:
            print(f"Error during prices fetching: {response.status_code}")
    else:
        print(f"Error during id fetching: {response.status_code}")

IndexError: list index out of range